# 量化交易入门 Vol.3：机器学习选股模型

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/qlora-sft-tutorial/blob/main/quant/03_ml_models.ipynb)

> **Kaggle 一键运行**：点击上方按钮 → 选择 **"CPU"** → Run All

## 本节你将学到

| 知识点 | 说明 |
|--------|------|
| **Panel 数据集构建** | 把多支股票×多天的因子数据整理成 ML 可用的格式 |
| **时间序列交叉验证** | 为什么金融数据不能用随机划分 |
| **LightGBM 选股** | 用梯度提升树预测截面相对收益 |
| **Rank IC 评估** | 评估模型预测质量的正确方式 |
| **分位数分析** | 模型能否区分好坏股票 |
| **特征重要性** | 哪些因子最有效 |

## 核心问题

在 Vol.2 中，我们手动等权合并了多个因子。  
本节的问题是：**机器学习能否自动找到更好的因子组合权重？**

In [ ]:
import subprocess, sys
pkgs = ["yfinance>=0.2.30", "pandas>=1.5.0", "numpy>=1.23.0",
        "matplotlib>=3.6.0", "seaborn>=0.12.0", "scipy>=1.9.0",
        "lightgbm>=3.3.0", "scikit-learn>=1.0.0"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("✅ 依赖安装完毕")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import lightgbm as lgb
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11,
                     'axes.titlesize': 12, 'axes.grid': True, 'grid.alpha': 0.3})

TICKERS = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'TSLA',
           'JPM', 'GS', 'JNJ', 'WMT', 'XOM', 'CVX', 'PG', 'KO', 'DIS']
START, END = '2018-01-01', '2024-01-01'

raw     = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False)
prices  = raw['Close'].dropna(how='all')[TICKERS]
volumes = raw['Volume'].dropna(how='all')[TICKERS]
highs   = raw['High'].dropna(how='all')[TICKERS]
lows    = raw['Low'].dropna(how='all')[TICKERS]
log_rets = np.log(prices / prices.shift(1))

print(f"数据加载完毕: {prices.shape[0]} 交易日 × {len(TICKERS)} 只股票")

## Step 1：构建特征矩阵（Feature Engineering）

我们将计算更丰富的因子集合作为 ML 模型的特征。  
共 15 个特征，覆盖 5 个因子族：

```
动量族（4 个）：  1m, 3m, 6m, 12-1m 动量
反转族（2 个）：  1w, 1m 反转
波动族（3 个）：  20d, 60d 波动率，日内振幅
成交量族（3 个）：成交量比，量价相关，成交量趋势
技术族（3 个）：  RSI, Bollinger %B, 价格加速度
```

**重要**：所有特征都做 `shift(1)` —— 只用昨天及之前的信息预测今天。

In [ ]:
def rank_cs(df):
    return df.rank(axis=1, pct=True).subtract(0.5)  # 截面排名，中心化到 [-0.5, 0.5]

def safe_shift(df, n=1):
    return df.shift(n)

features = {}

# ── 动量族 ────────────────────────────────────────────────
# 用截面排名：同一天内，这只股票比其他股票涨了多少？
for d, name in [(5,'mom_1w'), (21,'mom_1m'), (63,'mom_3m'), (126,'mom_6m')]:
    features[name] = rank_cs(safe_shift(prices.pct_change(d)))

# 12-1 月动量（去掉最近一个月避免短期反转）
features['mom_12m_skip1m'] = rank_cs(safe_shift(
    prices.pct_change(252) - prices.pct_change(21)))

# ── 反转族 ────────────────────────────────────────────────
# 取负：过去涨得多的 → 排名低（预期反转下跌）
for d, name in [(5,'rev_1w'), (21,'rev_1m')]:
    features[name] = rank_cs(safe_shift(-prices.pct_change(d)))

# ── 波动族 ────────────────────────────────────────────────
# 低波动因子：波动率低的股票，截面排名高
for d, name in [(20,'vol_20d'), (60,'vol_60d')]:
    vol = log_rets.rolling(d).std() * np.sqrt(252)
    features[name] = rank_cs(safe_shift(-vol))

# 日内振幅（high-low range）
intraday_range = (highs - lows) / prices
features['intraday_range'] = rank_cs(safe_shift(-intraday_range))

# ── 成交量族 ────────────────────────────────────────────────
# 量比：近期量 / 长期量（放量信号）
vol_ratio = volumes.rolling(5).mean() / volumes.rolling(60).mean()
features['vol_ratio'] = rank_cs(safe_shift(vol_ratio))

# 量价相关（过去 20 天价量相关系数，负相关=量价背离，看空）
vol_price_corr = pd.DataFrame(index=prices.index, columns=TICKERS, dtype=float)
for ticker in TICKERS:
    vol_price_corr[ticker] = prices[ticker].rolling(20).corr(volumes[ticker])
features['vol_price_corr'] = rank_cs(safe_shift(-vol_price_corr))

# 成交量趋势
vol_trend = volumes.rolling(5).mean() / volumes.rolling(20).mean()
features['vol_trend'] = rank_cs(safe_shift(vol_trend))

# ── 技术族 ────────────────────────────────────────────────
# RSI(14)，反向排名（低 RSI → 高因子值）
delta = prices.diff()
gain  = delta.clip(lower=0)
loss  = (-delta).clip(lower=0)
rs    = gain.rolling(14).mean() / loss.rolling(14).mean().replace(0, 1e-10)
rsi14 = 100 - 100 / (1 + rs)
features['rsi_14'] = rank_cs(safe_shift(100 - rsi14))

# Bollinger %B（价格在布林带中的位置）
ma20   = prices.rolling(20).mean()
std20  = prices.rolling(20).std()
boll_b = (prices - (ma20 - 2*std20)) / (4*std20.replace(0, 1e-10))  # 0=下轨, 0.5=中轨, 1=上轨
features['boll_b'] = rank_cs(safe_shift(-boll_b))  # 低 %B（超卖）→ 高因子

# 价格加速度（二阶动量）
ret5  = prices.pct_change(5)
accel = ret5 - ret5.shift(5)
features['price_accel'] = rank_cs(safe_shift(accel))

FEATURE_NAMES = list(features.keys())
print(f"共 {len(FEATURE_NAMES)} 个特征：")
for i, name in enumerate(FEATURE_NAMES):
    print(f"  [{i+1:2d}] {name}")

## Step 2：构建 Panel 数据集

把多支股票 × 多天的因子数据，整理成 ML 标准的二维表格格式：

```
每一行 = 一支股票在一天的快照
每一列 = 一个特征或标签

     date        stock   mom_1w   mom_1m  ...  label_10d
  2020-01-03    AAPL     0.62     0.71   ...    0.03
  2020-01-03    MSFT     0.45     0.53   ...   -0.01
  2020-01-03    NVDA     0.88     0.90   ...    0.08
  ...
  2020-01-06    AAPL     0.58     0.68   ...    0.02
  ...
```

**标签**：未来 10 个交易日的截面排名收益（Rank Return）  
用截面排名而非绝对收益的原因：去掉市场 beta，专注于选股能力

In [ ]:
HORIZON = 10  # 预测未来 10 个交易日

# 标签：未来 10 天的截面排名收益（去掉市场 beta）
future_ret_raw = log_rets.rolling(HORIZON).sum().shift(-HORIZON)
label = future_ret_raw.rank(axis=1, pct=True).subtract(0.5)  # 截面排名，[-0.5, 0.5]

# 堆叠成长表格（wide → long）：向量化实现，比逐日循环快 100x
print("构建 Panel 数据集...", end=' ')

stacked_features = {fname: feat.loc['2019-01-01':'2023-12-31'].stack()
                    for fname, feat in features.items()}
stacked_label = label.loc['2019-01-01':'2023-12-31'].stack()

panel = pd.concat({**stacked_features, 'label': stacked_label}, axis=1)
panel.index.names = ['date', 'ticker']
panel = panel.reset_index()
panel = panel.dropna(subset=FEATURE_NAMES + ['label'])
print(f"完成！共 {len(panel)} 行")
print(f"\n数据概览：")
print(panel.head(3).to_string())
print(f"\n日期范围: {panel['date'].min().date()} → {panel['date'].max().date()}")
print(f"平均每天样本数: {len(panel) / panel['date'].nunique():.1f} 只股票")

## Step 3：时间序列交叉验证 —— 金融数据为何不能随机划分？

### 绝对不能做的事

```python
# ❌ 错误做法：随机划分
from sklearn.model_selection import train_test_split
X_train, X_test = train_test_split(panel, test_size=0.2, random_state=42)
# 后果：用 2022 年的数据预测 2020 年 → 未来数据泄漏 → 虚假高收益
```

### 正确做法：Walk-Forward Validation（滚动验证）

```
时间轴 →

[训练 2019-2021] [测试 2022-Q1]
       [训练 2019-2022Q1] [测试 2022-Q2]
              [训练 2019-2022Q2] [测试 2022-Q3]
                     ...持续滚动...
```

这样保证：测试集的数据时间 > 训练集的所有数据时间

In [ ]:
# Walk-Forward 划分配置
TRAIN_END_DATES = [
    pd.Timestamp('2021-12-31'),  # 训练到2021年底 → 测试2022年Q1
    pd.Timestamp('2022-06-30'),  # 训练到2022年Q2 → 测试2022年Q3
    pd.Timestamp('2022-12-31'),  # 训练到2022年底 → 测试2023年Q1
    pd.Timestamp('2023-06-30'),  # 训练到2023年Q2 → 测试2023年Q3
]
TEST_MONTHS = 3  # 每次测试 3 个月

folds = []
for train_end in TRAIN_END_DATES:
    test_start = train_end + pd.Timedelta(days=1)
    test_end   = test_start + pd.DateOffset(months=TEST_MONTHS)
    
    train_mask = panel['date'] <= train_end
    test_mask  = (panel['date'] >= test_start) & (panel['date'] <= test_end)  # fix: >= not >
    
    if train_mask.sum() > 100 and test_mask.sum() > 10:
        folds.append({
            'train': panel[train_mask],
            'test':  panel[test_mask],
            'train_end':   train_end,
            'test_period': f"{test_start.date()}~{test_end.date()}"
        })

print(f"Walk-Forward 验证配置: {len(folds)} 个时间段")
print(f"{'折':<5} {'训练截止':>12} {'测试区间':>25} {'训练样本':>10} {'测试样本':>10}")
print("-" * 65)
for i, fold in enumerate(folds):
    print(f"Fold {i+1}  {fold['train_end'].date()!s:>12}  "
          f"{fold['test_period']:>25}  "
          f"{len(fold['train']):>10,}  "
          f"{len(fold['test']):>10,}")

# 可视化
fig, ax = plt.subplots(figsize=(12, 4))

for i, fold in enumerate(folds):
    train_dates = fold['train']['date']
    test_dates  = fold['test']['date']
    y = i * 1.1
    ax.barh(y, (train_dates.max() - train_dates.min()).days,
            left=train_dates.min(), height=0.8, color='steelblue', alpha=0.7, label='训练' if i==0 else '')
    ax.barh(y, (test_dates.max() - test_dates.min()).days,
            left=test_dates.min(), height=0.8, color='coral', alpha=0.9, label='测试' if i==0 else '')
    ax.text(test_dates.max() + pd.Timedelta(days=10), y, f'Fold {i+1}', va='center')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.set_yticks([])
ax.set_title('Walk-Forward 验证时间划分（蓝=训练，红=测试）\n每次测试集都在训练集时间之后，无未来数据泄漏')
ax.legend()
plt.tight_layout()
plt.savefig('walk_forward.png', bbox_inches='tight')
plt.show()

## Step 4：训练 LightGBM 模型

### 为什么用 LightGBM？

| 特性 | 说明 |
|------|------|
| **处理非线性** | 因子之间存在复杂的交互关系 |
| **自动特征选择** | 内置特征重要性，无需手动筛选 |
| **处理缺失值** | 不需要填充缺失值 |
| **训练速度快** | Histogram-based GBDT，比 XGBoost 快 |
| **量化领域实践多** | 被 Jane Street、AQR 等量化机构广泛使用 |

### 关键超参数

| 参数 | 推荐值 | 说明 |
|------|--------|------|
| `num_leaves` | 32-128 | 树的叶子数，越大越复杂 |
| `learning_rate` | 0.01-0.05 | 学习率，小=稳定但慢 |
| `n_estimators` | 100-500 | 树的数量 |
| `subsample` | 0.6-0.8 | 样本抽样比例（防过拟合） |
| `colsample_bytree` | 0.7-1.0 | 特征抽样比例 |

In [ ]:
lgb_params = {
    'objective': 'regression',
    'metric': 'mse',
    'num_leaves': 64,
    'learning_rate': 0.02,
    'n_estimators': 200,
    'subsample': 0.7,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'min_child_samples': 20,
    'verbose': -1,
    'random_state': 42,
}

def train_and_eval(fold_data):
    """训练一折并返回预测结果"""
    train, test = fold_data['train'], fold_data['test']
    
    X_train = train[FEATURE_NAMES]
    y_train = train['label']
    X_test  = test[FEATURE_NAMES]
    y_test  = test['label']
    
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)]
    )
    
    preds = model.predict(X_test)
    
    # Rank IC — pre-select columns to avoid pandas 2.2 DataFrameGroupBy.apply deprecation
    test_copy = test.copy()
    test_copy['pred'] = preds
    daily_ic = test_copy.groupby('date')[['pred', 'label']].apply(
        lambda g: spearmanr(g['pred'], g['label'])[0] if len(g) >= 3 else np.nan
    ).dropna()
    
    return {
        'model': model,
        'test_data': test_copy,
        'daily_ic': daily_ic,
        'mean_ic': daily_ic.mean(),
        'icir': daily_ic.mean() / daily_ic.std() if daily_ic.std() > 0 else 0,
        'feature_importance': pd.Series(
            model.feature_importances_, index=FEATURE_NAMES
        ).sort_values(ascending=False)
    }

print("开始 Walk-Forward 训练...")
fold_results = []
for i, fold in enumerate(folds):
    print(f"  Fold {i+1}: 训练截止 {fold['train_end'].date()}, 测试 {fold['test_period']}")
    result = train_and_eval(fold)
    fold_results.append(result)
    print(f"    Rank IC = {result['mean_ic']:.4f}, ICIR = {result['icir']:.3f}")

print("\n训练完成！")

## Step 5：模型评估 —— Rank IC 与分位数分析

In [ ]:
# 合并所有折的结果
all_test = pd.concat([r['test_data'] for r in fold_results], ignore_index=True)
all_ic   = pd.concat([r['daily_ic'] for r in fold_results]).sort_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 图1：每日 IC 时序
ax = axes[0][0]
ax.bar(all_ic.index, all_ic, alpha=0.4, color='steelblue', width=1)
ax.plot(all_ic.rolling(20).mean(), color='steelblue', lw=2, label='20日滚动均值')
ax.axhline(0, color='black', lw=1)
ax.axhline(all_ic.mean(), color='red', ls='--', lw=1.5,
           label=f'整体均值 = {all_ic.mean():.4f}')
ax.set_title('日 Rank IC（预测值 vs 实际收益排名相关）\n正值越多越好')
ax.set_ylabel('Rank IC')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# 图2：IC 分布直方图
ax2 = axes[0][1]
from scipy.stats import norm as sp_norm
ic_vals = all_ic.dropna()
ax2.hist(ic_vals, bins=40, density=True, alpha=0.6, color='steelblue', label='IC 分布')
xs = np.linspace(ic_vals.min(), ic_vals.max(), 200)
ax2.plot(xs, sp_norm.pdf(xs, ic_vals.mean(), ic_vals.std()), 'r-', lw=2, label='正态拟合')
ax2.axvline(0, color='black', lw=1)
ax2.axvline(ic_vals.mean(), color='red', ls='--', lw=1.5,
            label=f'均值={ic_vals.mean():.4f}')
ax2.set_title('IC 分布')
ax2.set_xlabel('Rank IC')
ax2.legend()

# 图3：分位数累计收益
ax3 = axes[1][0]
N_Q = 5
all_test['pred_quintile'] = all_test.groupby('date')['pred'].transform(
    lambda x: pd.qcut(x, N_Q, labels=False, duplicates='drop') + 1 if len(x) >= N_Q else np.nan
)

colors_q = ['#d73027', '#fc8d59', '#fee090', '#91bfdb', '#4575b4']
for q in range(1, N_Q + 1):
    q_data = all_test[all_test['pred_quintile'] == q].groupby('date')['label'].mean()
    q_cum  = (1 + q_data).cumprod()
    ax3.plot(q_cum.index, q_cum, color=colors_q[q-1], lw=2, label=f'Q{q}')

ax3.set_title('分位数累计收益\nQ1=预测最低组, Q5=预测最高组')
ax3.set_ylabel('Cumulative Return')
ax3.legend()
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# 图4：各分位组平均 label（截面排名收益）
ax4 = axes[1][1]
q_avg = all_test.groupby('pred_quintile')['label'].mean()
colors_b = ['#d73027', '#fc8d59', '#fee090', '#91bfdb', '#4575b4']
bars = ax4.bar([f'Q{int(q)}' for q in q_avg.index], q_avg.values,
               color=colors_b, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, q_avg.values):
    ax4.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
             f'{val:.4f}', ha='center', va='bottom', fontsize=11)
ax4.axhline(0, color='black', lw=1)
ax4.set_ylabel('平均标签值（截面排名收益）')
ax4.set_title('分位组平均收益\n单调递增=模型有效区分强弱股')

plt.suptitle(f'LightGBM 选股模型评估\n整体 Rank IC = {all_ic.mean():.4f}, ICIR = {all_ic.mean()/all_ic.std():.3f}',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('ml_evaluation.png', bbox_inches='tight')
plt.show()

print(f"\n模型汇总：")
print(f"  整体 Rank IC  = {all_ic.mean():.4f}")
print(f"  IC 标准差     = {all_ic.std():.4f}")
print(f"  ICIR          = {all_ic.mean()/all_ic.std():.3f}")
print(f"  IC>0 占比     = {(all_ic>0).mean()*100:.1f}%")

## Step 6：特征重要性分析 —— 哪些因子最有用？

In [ ]:
# 汇总所有折的特征重要性（取平均）
all_imp = pd.DataFrame([r['feature_importance'] for r in fold_results]).mean().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 左：特征重要性条形图
ax = axes[0]
colors = sns.color_palette('RdYlGn', len(all_imp))
bars = ax.barh(all_imp.index, all_imp.values, color=colors, edgecolor='white', height=0.7)
ax.set_xlabel('Feature Importance（平均跨折）')
ax.set_title('LightGBM 特征重要性\n（越高=模型越依赖该因子）')

for bar, val in zip(bars, all_imp.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2.,
            f'{val:.0f}', va='center', fontsize=9)

# 右：每折特征重要性热力图（稳定性）
ax2 = axes[1]
imp_matrix = pd.DataFrame(
    [r['feature_importance'].values for r in fold_results],
    columns=FEATURE_NAMES,
    index=[f'Fold {i+1}' for i in range(len(fold_results))]
)
# 归一化到 [0, 1]
imp_norm = (imp_matrix - imp_matrix.min()) / (imp_matrix.max() - imp_matrix.min() + 1e-10)
sns.heatmap(imp_norm.T, ax=ax2, cmap='YlOrRd', annot=True, fmt='.2f',
            linewidths=0.5, cbar_kws={'label': '重要性（归一化）'})
ax2.set_title('特征重要性跨折稳定性\n每行一个特征，每列一折\n稳定的因子=各折重要性一致')

plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

print("\n特征重要性排名（高→低）：")
all_imp_sorted = all_imp.sort_values(ascending=False)
for i, (feat, imp) in enumerate(all_imp_sorted.items(), 1):
    print(f"  [{i:2d}] {feat:<20s}: {imp:.1f}")

## Step 7：过拟合警报 —— 量化 ML 最常见的陷阱

量化中最危险的错误是：**在训练集上表现极好，实盘一塌糊涂**。

### 常见过拟合来源

| 来源 | 描述 | 防范 |
|------|------|------|
| **数据挖掘偏差** | 从大量因子中挑表现最好的，天然会挑到偶然好的 | 多因子结合，避免单因子优化 |
| **参数过拟合** | 超参数调得太好，只适合历史数据 | 简单参数 + Walk-Forward 验证 |
| **未来数据泄漏** | 不小心用了当天或未来数据 | 严格 shift(1) |
| **样本外 IC 下降** | 训练 IC 高，测试 IC 低 | 两者差距大则过拟合 |
| **机制变化** | 2020年前有效的策略，之后可能失效 | 用近期数据测试 |

In [ ]:
# 训练集 vs 测试集 IC 对比（检测过拟合）
print("过拟合检测：训练集 vs 测试集 IC")
print(f"{'折':<8} {'训练IC':>10} {'测试IC':>10} {'衰减':>10}")
print("-" * 42)

for i, (fold, result) in enumerate(zip(folds, fold_results)):
    train = fold['train']
    X_tr = train[FEATURE_NAMES]
    y_tr = train['label']
    preds_tr = result['model'].predict(X_tr)
    
    train_ic_list = []
    for date in train['date'].unique()[-60:]:  # 只看最近 60 个训练日
        mask = train['date'] == date
        g = train[mask]
        p = preds_tr[mask.values]
        if len(g) >= 3:
            ic, _ = spearmanr(p, g['label'])
            train_ic_list.append(ic)
    
    train_ic = np.mean(train_ic_list) if train_ic_list else 0
    test_ic  = result['mean_ic']
    decay    = (train_ic - test_ic) / (abs(train_ic) + 1e-10) * 100
    
    status = "⚠️  过拟合" if decay > 30 else "✅  正常"
    print(f"Fold {i+1}   {train_ic:>10.4f} {test_ic:>10.4f} {decay:>8.1f}%  {status}")

print("\n建议：")
print("  训练→测试 IC 衰减 < 20% 认为正常")
print("  衰减 > 50% 需要减少特征/增加正则")
print("  真实市场中 IC 0.03-0.05 已是不错的模型")

In [ ]:
# 保存预测结果供 Vol.4 回测使用
all_test['date_str'] = all_test['date'].astype(str)
all_test[['date', 'ticker', 'pred', 'label']].to_csv('ml_predictions.csv', index=False)
print("预测结果已保存至 ml_predictions.csv")
print(f"形状: {all_test.shape}")
print(all_test[['date', 'ticker', 'pred', 'label']].head(10).to_string())

## 本节总结

```
ML 选股模型构建流程

因子特征（Vol.2）
    │
    ├── Panel 数据集构建（每行=一只股票在一天）
    │       ├── 特征：因子截面排名
    │       └── 标签：未来 N 天截面排名收益
    │
    ├── Walk-Forward 时间序列验证（不可随机划分！）
    │
    ├── LightGBM 训练
    │       ├── 自动学习因子权重
    │       └── 处理非线性因子交互
    │
    ├── 评估指标
    │       ├── Rank IC（预测值 vs 实际收益排名相关）
    │       ├── ICIR（IC 信噪比）
    │       └── 分位数收益（单调性）
    │
    └── 过拟合检测（训练 vs 测试 IC 衰减）
```

### LightGBM 量化经验总结

| 经验 | 说明 |
|------|------|
| 标签用截面排名 | 不受绝对收益量纲影响 |
| n_estimators 不要太大 | 200-300 通常已够，避免过拟合 |
| subsample + colsample | 防过拟合核心参数 |
| Walk-Forward 必须做 | 没有这个，IC 没有意义 |
| 实际 IC 0.03 就不错 | 市场噪音太大，完美预测不可能 |

## 下一步：Vol.4 回测基础

有了 ML 模型的预测信号，下一节将学习如何把信号转换成**可交易的策略**，并进行**回测**评估真实盈利能力。

---
*Vol.3 完 | 课程：量化交易从入门到 qlib*